In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, accuracy_score, log_loss
import warnings
from tqdm import tqdm
import os
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/IrrigationNeed/')

In [23]:
irrigation_data = pd.read_csv("train.csv", index_col = 0)
irrigation_data

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High


In [34]:
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
transformer = ColumnTransformer(transformers=[('ohe', ohe, make_column_selector(dtype_include=object))], remainder='passthrough', verbose_feature_names_out=False).set_output(transform = 'pandas')

In [25]:
irrigation_data.isna().sum().sum()

0

In [35]:
X,y = irrigation_data.drop('Irrigation_Need',axis=1), irrigation_data['Irrigation_Need']

In [36]:
le = LabelEncoder()
y = le.fit_transform(y)

In [37]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y)

In [38]:
X_train_ohe = transformer.fit_transform(X_train)
X_test_ohe = transformer.transform(X_test)

In [39]:
ss = StandardScaler().set_output(transform = 'pandas')
X_train_scaled = ss.fit_transform(X_train_ohe)
X_test_scaled = ss.transform(X_test_ohe)

In [16]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            xgb = XGBClassifier(random_state=26, n_estimators=n, learning_rate=r, max_depth=d)
            xgb.fit(X_train_scaled, y_train)
            y_pred = xgb.predict(X_test_scaled)
            y_pred_proba = xgb.predict_proba(X_test_scaled)
            scores.append([n, r, d,log_loss(y_test, y_pred_proba), accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "Log Loss", "Accuracy"])
df_scores.sort_values(["Accuracy", "Log Loss"], ascending = [False, True])

100%|████████████████████████████████████████████| 5/5 [36:23<00:00, 436.61s/it]


,Number Of Estimators,Learning Rate,Depth,Log Loss,Accuracy
663,100,0.372472,5,0.057257,0.984706
509,75,0.341441,6,0.057672,0.984698
513,75,0.372472,5,0.057844,0.984675
673,100,0.434534,5,0.057176,0.984659
659,100,0.341441,6,0.057440,0.984659
...,...,...,...,...,...
600,100,0.000100,2,1.091422,0.751635
450,75,0.000100,2,1.093209,0.751635
300,50,0.000100,2,1.095002,0.751635
150,25,0.000100,2,1.096803,0.751635


In [17]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            cb = CatBoostClassifier(random_state=26, n_estimators=n, learning_rate=r, max_depth=d, verbose=0)
            cb.fit(X_train_scaled, y_train)
            y_pred = cb.predict(X_test_scaled)
            y_pred_proba = cb.predict_proba(X_test_scaled)
            scores.append([n, r, d,log_loss(y_test, y_pred_proba), accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "Log Loss", "Accuracy"])
df_scores.sort_values(["Accuracy", "Log Loss"], ascending = [False, True])

100%|████████████████████████████████████████████| 5/5 [31:55<00:00, 383.16s/it]


,Number Of Estimators,Learning Rate,Depth,Log Loss,Accuracy
542,75,0.558659,4,0.060823,0.984683
672,100,0.434534,4,0.060630,0.984675
649,100,0.279379,6,0.060450,0.984659
692,100,0.558659,4,0.060107,0.984651
682,100,0.496597,4,0.060241,0.984651
...,...,...,...,...,...
600,100,0.000100,2,1.089316,0.744722
450,75,0.000100,2,1.091619,0.744722
300,50,0.000100,2,1.093935,0.744722
150,25,0.000100,2,1.096265,0.744722


In [18]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            cb = LGBMClassifier(random_state=26, n_estimators=n, learning_rate=r, max_depth=d, verbose=-1)
            cb.fit(X_train_scaled, y_train)
            y_pred = cb.predict(X_test_scaled)
            y_pred_proba = cb.predict_proba(X_test_scaled)
            scores.append([n, r, d,log_loss(y_test, y_pred_proba), accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "Log Loss", "Accuracy"])
df_scores.sort_values(["Accuracy", "Log Loss"], ascending = [False, True])

100%|████████████████████████████████████████████| 5/5 [30:18<00:00, 363.69s/it]


,Number Of Estimators,Learning Rate,Depth,Log Loss,Accuracy
652,100,0.310410,4,0.057793,0.984762
647,100,0.279379,4,0.058097,0.984746
633,100,0.186286,5,0.057993,0.984738
637,100,0.217317,4,0.059175,0.984722
502,75,0.310410,4,0.058642,0.984714
...,...,...,...,...,...
581,75,0.806907,3,16.035707,0.555103
734,100,0.806907,6,16.857273,0.532310
576,75,0.775876,3,18.171723,0.495841
726,100,0.775876,3,18.171723,0.495841


In [44]:
X_ohe = transformer.fit_transform(X)
X_scaled = ss.fit_transform(X_ohe)

In [45]:
bm = XGBClassifier(random_state = 26, n_estimators = 100, max_depth = 5, learning_rate = 0.372472)

In [46]:
bm.fit(X_scaled, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.372472,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [40]:
tst = pd.read_csv("test.csv", index_col = 0)
tst

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,Yes,43.49,East
899996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,Yes,92.03,West
899997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,Yes,34.02,East


In [41]:
tst_ohe = transformer.transform(tst)

In [42]:
tst_ss = ss.transform(tst_ohe)
tst_ss

,Soil_Type_Loamy,Soil_Type_Sandy,Soil_Type_Silt,Crop_Type_Maize,Crop_Type_Potato,Crop_Type_Rice,Crop_Type_Sugarcane,Crop_Type_Wheat,Crop_Growth_Stage_Harvest,Crop_Growth_Stage_Sowing,...,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm
id,,,,,,,,,,,,,,,,,,,,,
630000,-0.575074,-0.599664,1.803017,2.245232,-0.440626,-0.452315,-0.457236,-0.441695,-0.601967,1.808412,...,-0.678538,-0.910583,1.120205,-1.063516,-1.587713,0.116065,-1.057012,-1.296494,1.440660,-0.431814
630001,-0.575074,-0.599664,-0.554626,-0.445388,-0.440626,-0.452315,-0.457236,-0.441695,-0.601967,1.808412,...,-1.674225,0.702086,1.592967,-0.674912,0.838016,-1.445480,-0.146831,0.966773,-0.330996,-0.170503
630002,-0.575074,1.667600,-0.554626,-0.445388,-0.440626,-0.452315,-0.457236,2.264006,-0.601967,1.808412,...,-0.656560,0.100752,-0.938932,-0.014865,-0.060665,-1.495637,0.958389,-1.361510,-1.044876,-1.234142
630003,-0.575074,-0.599664,-0.554626,2.245232,-0.440626,-0.452315,-0.457236,-0.441695,1.661220,-0.552971,...,0.993557,-0.254582,-1.254106,1.710074,-1.293055,-0.409735,-0.411884,-1.496814,-1.244099,1.188900
630004,1.738905,-0.599664,-0.554626,-0.445388,-0.440626,-0.452315,-0.457236,-0.441695,-0.601967,1.808412,...,1.325656,-1.047250,0.384799,1.633513,-0.460304,-0.228874,-1.702141,-0.820294,1.156057,-1.428885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,-0.575074,1.667600,-0.554626,-0.445388,2.269497,-0.452315,-0.457236,-0.441695,-0.601967,-0.552971,...,0.890997,-0.664583,0.878572,0.727543,0.534736,1.409938,-0.211844,-0.062943,-1.087567,-0.548309
899996,1.738905,-0.599664,-0.554626,-0.445388,-0.440626,-0.452315,-0.457236,2.264006,-0.601967,-0.552971,...,0.479536,-0.199916,-0.739322,0.064016,-0.831036,1.403022,-1.182037,-0.836109,0.880940,0.868903
899997,1.738905,-0.599664,-0.554626,2.245232,-0.440626,-0.452315,-0.457236,-0.441695,-0.601967,-0.552971,...,-1.604630,1.740754,0.164177,-0.418549,0.168569,-0.759306,1.463489,-0.069972,-1.099425,-0.824803


In [47]:
tst["Irrigation_Need"] = le.inverse_transform(bm.predict(tst_ss))
tst

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West,Low
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South,Medium
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East,Low
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North,Low
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,Yes,43.49,East,Medium
899996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,Yes,92.03,West,Low
899997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,Yes,34.02,East,Medium


In [48]:
submit = pd.read_csv("sample_submission.csv")
submit

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Low
269996,899996,Low
269997,899997,Low
269998,899998,Low


In [49]:
submit["Irrigation_Need"] = le.inverse_transform(bm.predict(tst_ss))
submit

,id,Irrigation_Need
0,630000,Low
1,630001,Medium
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Medium
269996,899996,Low
269997,899997,Medium
269998,899998,Low


In [50]:
submit.to_csv("KaggleSubmissionXGB.csv", index = False)